# GPT Hierarchical Guideline Generation

Use the prompt helpers in `src/data_handlers/hierarchical_guideline_prompt.py` to generate hierarchical NER annotation guidelines with the OpenAI API. The notebook supports a single-entity smoke test and resumable batch generation from `data/pileNER/pileNER_391_3pos_2neg_examples.json`.

## Setup

Set `OPENAI_API_KEY` in your environment before running API cells. The notebook defaults to an OpenAI-compatible proxy endpoint through `BASE_URL`; override it with `OPENAI_BASE_URL` if needed. The default batch limit is intentionally tiny; set `MAX_ENTITY_TYPES = None` only when you are ready to run all 391 entity types.

In [16]:
from pathlib import Path
import json
import os
import sys
import time
from datetime import datetime

from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "data_handlers" / "hierarchical_guideline_prompt.py").exists():
            return path
    raise RuntimeError("Could not find the SLIMER repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_handlers.hierarchical_guideline_prompt import (  # noqa: E402
    GUIDELINE_FIELDS,
    build_hierarchical_guideline_messages,
    parse_guideline_json,
)

print(f"Repository root: {REPO_ROOT}")

Repository root: D:\Projects\SLIMER


In [26]:
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://apix.ai-gaochao.cn/v1")
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 1200

INPUT_PATH = REPO_ROOT / "data" / "pileNER" / "pileNER_391_3pos_2neg_examples.json"
OUTPUT_PATH = REPO_ROOT / "data" / "pileNER" / "pileNER_391_hierarchical_guidelines.json"
FAILURE_PATH = REPO_ROOT / "data" / "pileNER" / "pileNER_391_hierarchical_guideline_failures.json"

INCLUDE_FEW_SHOT = False  # Set to True to include few-shot examples in the prompt.
MAX_ENTITY_TYPES = 391  # Set to None to run all entity types after the smoke test.
SKIP_EXISTING = True
OVERWRITE = False
REQUEST_SLEEP_SECONDS = 1.0
MAX_RETRIES = 3

GUIDELINE_RESPONSE_FORMAT = {
    "type": "json_schema",
    "name": "hierarchical_guideline",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "entity_type": {"type": "string"},
            "definition": {"type": "string"},
            "include_rules": {"type": "array", "items": {"type": "string"}},
            "exclude_rules": {"type": "array", "items": {"type": "string"}},
            "boundary_rules": {"type": "array", "items": {"type": "string"}},
            "confusable_type_rules": {"type": "array", "items": {"type": "string"}},
        },
        "required": GUIDELINE_FIELDS,
        "additionalProperties": False,
    },
}

print(f"Model: {MODEL}")
print(f"Base URL: {BASE_URL}")
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

Model: gpt-5.4
Base URL: https://apix.ai-gaochao.cn/v1
Input: D:\Projects\SLIMER\data\pileNER\pileNER_391_3pos_2neg_examples.json
Output: D:\Projects\SLIMER\data\pileNER\pileNER_391_hierarchical_guidelines.json


## Load Examples

The input file is grouped by entity type. Each item should contain `positive_examples` and may contain `negative_examples`; both are passed into `build_hierarchical_guideline_messages()` without custom prompt rewriting.

In [18]:
def load_examples(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict) or not data:
        raise ValueError(f"Expected a non-empty JSON object at {path}")

    bad_entity_types = []
    for entity_type, item in data.items():
        if not isinstance(item, dict) or not isinstance(item.get("positive_examples"), list):
            bad_entity_types.append(entity_type)
    if bad_entity_types:
        preview = bad_entity_types[:10]
        raise ValueError(f"Missing positive_examples for entity types: {preview}")
    return data


examples_by_type = load_examples(INPUT_PATH)
entity_types = list(examples_by_type)

print(f"Loaded {len(entity_types)} entity types.")
print("First 10 entity types:", entity_types[:10])

Loaded 391 entity types.
First 10 entity types: ['abbreviation', 'ability', 'acronym', 'action', 'activity', 'address', 'adjective', 'age', 'age group', 'agreement']


In [19]:
PREVIEW_ENTITY_TYPE = entity_types[0]
preview_item = examples_by_type[PREVIEW_ENTITY_TYPE]
preview_messages = build_hierarchical_guideline_messages(
    entity_type=PREVIEW_ENTITY_TYPE,
    positive_examples=preview_item["positive_examples"],
    negative_or_boundary_examples=preview_item.get("negative_examples"),
    include_few_shot=INCLUDE_FEW_SHOT,
)

print(f"Preview entity type: {PREVIEW_ENTITY_TYPE}")
print(f"Message count: {len(preview_messages)}")
print(preview_messages[-1]["content"])

Preview entity type: abbreviation
Message count: 2
We are constructing hierarchical annotation guidelines for a named entity type.

Your output must use these JSON fields:
- entity_type: the entity type name.
- definition: the semantic definition of the entity type.
- include_rules: rules describing what should be annotated.
- exclude_rules: rules describing what should not be annotated.
- boundary_rules: rules describing span start/end decisions.
- confusable_type_rules: rules distinguishing this type from confusable entity types.

Named Entity:
abbreviation

Positive examples:
[
  {
    "sentence": "The Nationals are thinking about Zimmerman having an MRI on Friday.",
    "entities": [
      "MRI"
    ]
  },
  {
    "sentence": "Our institution had an outbreak of multi-drug-resistant Acinetobacter (MDRA) in 2011.",
    "entities": [
      "MDRA"
    ]
  },
  {
    "sentence": "Following the 2007 successful IPO, the company is also listed on Nasdaq OMX Stockholm (NET-B).",
    "entiti

## API Helpers

These helpers use OpenAI Structured Outputs and still validate with the local parser before saving.

In [20]:
api_key = os.getenv("OPENAI_API_KEY","sk-7t5lOP05xHHP6d3sBaCc8b1bF14a46F5820576F7106923C0")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running API cells.")

client = OpenAI(api_key=api_key, base_url=BASE_URL)


def extract_response_text(response) -> str:
    output_text = getattr(response, "output_text", None)
    if output_text:
        return output_text

    text_parts = []
    for output_item in getattr(response, "output", []) or []:
        for content_item in getattr(output_item, "content", []) or []:
            text = getattr(content_item, "text", None)
            if text:
                text_parts.append(text)
    if text_parts:
        return "".join(text_parts)

    raise ValueError("Could not extract text from the OpenAI response.")


def order_guideline_fields(guideline: dict) -> dict:
    """Return a guideline dict in the canonical field order used by the prompt."""
    parsed = parse_guideline_json(guideline)
    return {field: parsed[field] for field in GUIDELINE_FIELDS}


def generate_guideline(entity_type: str, item: dict) -> dict:
    messages = build_hierarchical_guideline_messages(
        entity_type=entity_type,
        positive_examples=item["positive_examples"],
        negative_or_boundary_examples=item.get("negative_examples"),
        include_few_shot=INCLUDE_FEW_SHOT,
    )

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=messages,
                temperature=TEMPERATURE,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                text={"format": GUIDELINE_RESPONSE_FORMAT},
            )
            guideline = order_guideline_fields(json.loads(extract_response_text(response)))
            if guideline["entity_type"].strip().lower() != entity_type.strip().lower():
                print(
                    "Warning: returned entity_type differs from requested entity type: "
                    f"{guideline['entity_type']!r} != {entity_type!r}"
                )
            return guideline
        except Exception as exc:  # Keep notebook-friendly retry behavior.
            last_error = exc
            if attempt < MAX_RETRIES:
                wait_seconds = 2 ** (attempt - 1)
                print(f"Attempt {attempt} failed for {entity_type}: {exc}. Retrying in {wait_seconds}s.")
                time.sleep(wait_seconds)

    raise RuntimeError(f"Failed to generate guideline for {entity_type}: {last_error}") from last_error

## Single Test

Run this before batch generation. It makes exactly one API request.

In [21]:
SINGLE_ENTITY_TYPE = PREVIEW_ENTITY_TYPE

single_guideline = generate_guideline(SINGLE_ENTITY_TYPE, examples_by_type[SINGLE_ENTITY_TYPE])
print(json.dumps(single_guideline, ensure_ascii=False, indent=2))

{
  "entity_type": "abbreviation",
  "definition": "A shortened form, initialism, acronym, code-like shorthand, or ticker-style label used in place of a longer name or expression.",
  "include_rules": [
    "Annotate standalone abbreviations, acronyms, and initialisms that compress a longer expression into a shorter token, such as 'MRI' and 'MDRA'.",
    "Annotate shorthand labels introduced after a full form in parentheses, e.g., 'multi-drug-resistant Acinetobacter (MDRA)'.",
    "Annotate code-like abbreviated identifiers used as shortened names for organizations, products, or listings, such as ticker-style forms like 'NET-B'.",
    "Annotate abbreviations composed of capital letters, and also those containing internal hyphens or digits when they function as a shortened form.",
    "Annotate the abbreviated mention whether or not the full expansion appears elsewhere in the same sentence, as long as the token itself is a recognized shorthand form in context."
  ],
  "exclude_rules": [

## Batch Generation

Results are written incrementally to `data/pileNER/pileNER_391_hierarchical_guidelines.json`. Existing valid entries are skipped by default so the run can resume after interruptions.

In [27]:
def atomic_write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    tmp_path.replace(path)


def load_existing_guidelines(path: Path) -> dict:
    if OVERWRITE or not path.exists():
        return {}
    existing = json.loads(path.read_text(encoding="utf-8"))
    valid = {}
    for entity_type, guideline in existing.items():
        try:
            valid[entity_type] = order_guideline_fields(guideline)
        except Exception as exc:
            print(f"Ignoring invalid existing guideline for {entity_type}: {exc}")
    return valid


batch_entity_types = entity_types if MAX_ENTITY_TYPES is None else entity_types[:MAX_ENTITY_TYPES]
guidelines = load_existing_guidelines(OUTPUT_PATH)
failures = {}
if OUTPUT_PATH.exists() and guidelines:
    atomic_write_json(OUTPUT_PATH, guidelines)
    print(f"Normalized existing guideline field order: {OUTPUT_PATH}")


started_at = datetime.now().isoformat(timespec="seconds")
print(f"Batch started at {started_at}")
print(f"Target entity types this run: {len(batch_entity_types)}")
print(f"Existing valid guidelines: {len(guidelines)}")

for index, entity_type in enumerate(batch_entity_types, start=1):
    if SKIP_EXISTING and entity_type in guidelines:
        print(f"[{index}/{len(batch_entity_types)}] skip existing: {entity_type}")
        continue

    print(f"[{index}/{len(batch_entity_types)}] generating: {entity_type}")
    try:
        guideline = generate_guideline(entity_type, examples_by_type[entity_type])
        guidelines[entity_type] = guideline
        atomic_write_json(OUTPUT_PATH, guidelines)
        print(f"  saved {entity_type}; total saved: {len(guidelines)}")
    except Exception as exc:
        failures[entity_type] = str(exc)
        atomic_write_json(FAILURE_PATH, failures)
        print(f"  failed {entity_type}: {exc}")

    if REQUEST_SLEEP_SECONDS:
        time.sleep(REQUEST_SLEEP_SECONDS)

print(f"Done. Saved {len(guidelines)} valid guidelines to {OUTPUT_PATH}")
if failures:
    print(f"Failures saved to {FAILURE_PATH}: {len(failures)}")

Normalized existing guideline field order: D:\Projects\SLIMER\data\pileNER\pileNER_391_hierarchical_guidelines.json
Batch started at 2026-05-06T00:35:34
Target entity types this run: 391
Existing valid guidelines: 10
[1/391] skip existing: abbreviation
[2/391] skip existing: ability
[3/391] skip existing: acronym
[4/391] skip existing: action
[5/391] skip existing: activity
[6/391] skip existing: address
[7/391] skip existing: adjective
[8/391] skip existing: age
[9/391] skip existing: age group
[10/391] skip existing: agreement
[11/391] generating: aircraft
  saved aircraft; total saved: 11
[12/391] generating: amino acid
  saved amino acid; total saved: 12
[13/391] generating: anatomical entity
  saved anatomical entity; total saved: 13
[14/391] generating: anatomical structure
  saved anatomical structure; total saved: 14
[15/391] generating: anatomy
  saved anatomy; total saved: 15
[16/391] generating: animal
  saved animal; total saved: 16
[17/391] generating: antibody
  saved ant

In [ ]:
saved_guidelines = load_existing_guidelines(OUTPUT_PATH)
print(f"Valid saved guidelines: {len(saved_guidelines)}")

if saved_guidelines:
    first_entity_type = next(iter(saved_guidelines))
    print(f"Sample entity type: {first_entity_type}")
    print(json.dumps(saved_guidelines[first_entity_type], ensure_ascii=False, indent=2))

if FAILURE_PATH.exists():
    saved_failures = json.loads(FAILURE_PATH.read_text(encoding="utf-8"))
    print(f"Saved failures: {len(saved_failures)}")
    print(dict(list(saved_failures.items())[:5]))